# 🔬 Conference Room Quality — Model Analysis

Deep-dive into model performance:
1. Residual analysis for each model
2. Predicted vs. actual scatter plots
3. Feature importance comparison
4. Error analysis — which images are hardest to score?

> Run `python src/embed.py` and `python src/score.py` before this notebook.

In [ ]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

plt.rcParams.update({
    'figure.facecolor': '#0F0F1A',
    'axes.facecolor':   '#1A1A2E',
    'axes.edgecolor':   '#2D2D4E',
    'axes.labelcolor':  '#E8E8F0',
    'xtick.color':      '#E8E8F0',
    'ytick.color':      '#E8E8F0',
    'text.color':       '#E8E8F0',
    'grid.color':       '#2D2D4E',
    'grid.alpha':       0.4,
})
PALETTE = ['#6C63FF', '#43D3A6', '#FFC857', '#FF6584']

print('Ready. ROOT =', ROOT)

## 1. Load Models & Data

In [ ]:
import yaml
with open(ROOT / 'config.yaml') as f:
    config = yaml.safe_load(f)

target_col = config['model']['target_column']

# Out-of-fold predictions written by `python src/score.py`: every prediction comes from a
# model that never saw that image. (The saved models are refit on all data, so predicting
# with them here would score them on their own training set.)
feat_df = pd.read_csv(ROOT / 'features' / 'features.csv')
oof_df  = pd.read_csv(ROOT / 'outputs' / 'oof_predictions.csv')
merged  = feat_df.merge(oof_df, on='filename', how='inner')

feature_cols = [c for c in feat_df.columns if c != 'filename']
y = merged[target_col].astype(float)

# Two YOLO-feature models and the two CLIP-embedding models
model_names = ['ridge_regression', 'xgboost', 'clip_ridge', 'hybrid_ridge']
preds = {name: np.clip(merged[name].values, 1, 5) for name in model_names if name in merged}

# Fitted tree models (used only for feature importances below)
models = {}
for name in ['random_forest', 'xgboost']:
    path = ROOT / 'models' / target_col / f'{name}.joblib'
    if path.exists():
        models[name] = joblib.load(path)['model']
    else:
        print(f'Model not found: {path}')

best_name = min(preds, key=lambda n: mean_absolute_error(y, preds[n]))
print(f'{len(merged)} images | OOF predictions for {list(preds)} | best by MAE: {best_name}')

## 2. Predicted vs. Actual

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, (name, y_pred), color in zip(axes, preds.items(), PALETTE):
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    pearson_r, _ = pearsonr(y, y_pred)

    ax.scatter(y, y_pred, alpha=0.75, color=color, s=70, edgecolors='none')
    lo, hi = min(y.min(), 1) - 0.2, max(y.max(), 5) + 0.2
    ax.plot([lo, hi], [lo, hi], '--', color='white', alpha=0.5, lw=1.5, label='Ideal')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel('Actual Score', fontsize=9)
    ax.set_ylabel('Predicted Score', fontsize=9)
    ax.set_title(
        f'{name.replace("_", " ").title()}\nMAE={mae:.3f}  R²={r2:.3f}  r={pearson_r:.3f}',
        fontsize=10, fontweight='bold'
    )
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Predicted vs. Actual — out-of-fold', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_predicted_vs_actual_detail.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_pred_best = preds[best_name]
residuals = y.values - y_pred_best

# Residuals vs fitted
ax1 = axes[0]
ax1.scatter(y_pred_best, residuals, alpha=0.75, color='#6C63FF', s=60, edgecolors='none')
ax1.axhline(0, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.set_xlabel('Predicted Score', fontsize=10)
ax1.set_ylabel('Residual (Actual − Predicted)', fontsize=10)
ax1.set_title(f'Residuals vs. Fitted ({best_name})', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Residual histogram
ax2 = axes[1]
ax2.hist(residuals, bins=20, color='#43D3A6', edgecolor='none', alpha=0.85)
ax2.axvline(0, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.axvline(residuals.mean(), color='#FFC857', linestyle='-', linewidth=2,
            label=f'Mean={residuals.mean():.3f}')
ax2.set_xlabel('Residual', fontsize=10)
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title(f'Residual Distribution ({best_name})', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Residuals — Mean: {residuals.mean():.4f}  Std: {residuals.std():.4f}')
print(f'Max over-prediction: {residuals.min():.3f}  Max under-prediction: {residuals.max():.3f}')

## 4. Feature Importance — XGBoost vs Random Forest

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
tree_models = {k: v for k, v in models.items() if k in ['xgboost', 'random_forest']}

for ax, (name, estimator), color in zip(axes, tree_models.items(), ['#6C63FF', '#43D3A6']):
    raw = estimator.steps[-1][1] if hasattr(estimator, 'steps') else estimator
    if not hasattr(raw, 'feature_importances_'):
        continue
    importances = raw.feature_importances_
    sorted_idx = np.argsort(importances)
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(importances)))
    ax.barh([feature_cols[i] for i in sorted_idx], importances[sorted_idx],
            color=[colors[i] for i in range(len(sorted_idx))], edgecolor='none')
    ax.set_title(f'{name.replace("_"," ").title()} Feature Importances',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance', fontsize=9)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hardest Images to Score (Largest Errors)

In [ ]:
test_merged = merged.copy()
test_merged['y_pred'] = y_pred_best
test_merged['abs_error'] = (test_merged[target_col] - test_merged['y_pred']).abs()
hardest = test_merged.nlargest(5, 'abs_error')[['filename', target_col, 'y_pred', 'abs_error']]

print(f'Images with Largest Prediction Error ({best_name}, out-of-fold):')
print(hardest.to_string(index=False))

# Optionally display the hardest images
images_dir = ROOT / 'data' / 'images'
fig, axes = plt.subplots(1, min(5, len(hardest)), figsize=(16, 4))
if not hasattr(axes, '__iter__'):
    axes = [axes]

for ax, (_, row) in zip(axes, hardest.iterrows()):
    img_path = images_dir / row['filename']
    if img_path.exists():
        from PIL import Image as PILImage
        img = PILImage.open(img_path).resize((320, 240))
        ax.imshow(img)
        ax.set_title(
            f"{row['filename'][:20]}\n"
            f"True={row[target_col]:.1f} | Pred={row['y_pred']:.1f} | Err={row['abs_error']:.2f}",
            fontsize=8, color='#E8E8F0'
        )
    ax.axis('off')

fig.suptitle('Hardest Images to Score', fontsize=12, fontweight='bold', color='#E8E8F0')
fig.patch.set_facecolor('#0F0F1A')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_hardest_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cross-Validated MAE by Target

In [ ]:
# Cross-validated MAE for every model and target (from outputs/cv_results.csv, written by score.py)
cv = pd.read_csv(ROOT / 'outputs' / 'cv_results.csv')
pivot = cv.pivot(index='model', columns='target', values='MAE_mean')
order = ['dummy_mean', 'linear_regression', 'ridge_regression', 'random_forest', 'xgboost', 'clip_ridge', 'hybrid_ridge']
pivot = pivot.loc[[m for m in order if m in pivot.index]]

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot.bar(ax=ax, color=PALETTE, edgecolor='none', alpha=0.85)
ax.set_ylabel('CV MAE (lower is better)', fontsize=10)
ax.set_xlabel('')
ax.set_title('Repeated 5x5-fold CV MAE per model and target', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, facecolor='#1A1A2E', edgecolor='#2D2D4E')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_cv_by_target.png', dpi=150, bbox_inches='tight')
plt.show()
pivot.round(3)